# Prediction exploratoire du reachat

Ce notebook construit un premier modele pour estimer quels produits pourraient apparaitre dans la prochaine commande d'un client. Les candidats sont ses produits deja achetes, completes par les produits les plus frequents dans `prior`. Les produits `train` servent de reponse attendue.

L'evaluation est faite sur un echantillon reproductible de commandes `train`, separe par commande en entrainement, validation et test. La couverture mesure la part des produits attendus presents dans l'historique ou parmi les produits populaires ajoutes.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidats = [Path.cwd(), *Path.cwd().parents]
PROJET = next((p for p in candidats if (p / "data" / "raw").is_dir()), None)
if PROJET is None:
    raise FileNotFoundError("Racine du projet introuvable : data/raw est absent.")

DONNEES = PROJET / "data" / "processed"
for nom in ["orders.csv", "order_products__prior.csv", "order_products__train.csv", "products.csv"]:
    if not (DONNEES / nom).is_file():
        raise FileNotFoundError(
            f"Fichier prepare absent : {DONNEES / nom}. Executer d'abord le notebook de nettoyage."
        )

GRAINE = 42
NOMBRE_COMMANDES_CIBLE = 20_000
TAILLE_MORCEAU = 500_000
COLONNES_COMMANDES = [
    "order_id", "user_id", "eval_set", "order_number", "order_dow",
    "order_hour_of_day", "days_since_prior_order",
]
print(f"Projet : {PROJET}")
print(f"Donnees preparees : {DONNEES}")

Projet : c:\Users\mouft\Desktop\data\data_m2
Donnees preparees : c:\Users\mouft\Desktop\data\data_m2\data\processed


## 1. Choisir les commandes a predire

On tire au sort jusqu'a 20 000 commandes `train`. Pour chaque client retenu, seules ses commandes `prior` servent a construire les caracteristiques. Les produits de sa commande `train` servent uniquement a creer la reponse attendue (1 si le produit est present, 0 sinon).

In [2]:
parties_train = []
for morceau in pd.read_csv(
    DONNEES / "orders.csv", usecols=COLONNES_COMMANDES, chunksize=TAILLE_MORCEAU
):
    selection = morceau.loc[morceau["eval_set"].eq("train")]
    if not selection.empty:
        parties_train.append(selection)

commandes_train = pd.concat(parties_train, ignore_index=True)
if commandes_train["user_id"].duplicated().any():
    raise ValueError("Plusieurs commandes train pour un meme client : revoir la separation des donnees.")
if len(commandes_train) < 100:
    raise ValueError("Pas assez de commandes train pour entrainer et evaluer un modele.")

commandes_cibles = commandes_train.sample(
    n=min(NOMBRE_COMMANDES_CIBLE, len(commandes_train)),
    random_state=GRAINE,
).reset_index(drop=True)
utilisateurs_cibles = set(commandes_cibles["user_id"].tolist())

parties_prior = []
for morceau in pd.read_csv(
    DONNEES / "orders.csv", usecols=COLONNES_COMMANDES, chunksize=TAILLE_MORCEAU
):
    selection = morceau.loc[
        morceau["eval_set"].eq("prior") & morceau["user_id"].isin(utilisateurs_cibles),
        ["order_id", "user_id", "order_number"],
    ]
    if not selection.empty:
        parties_prior.append(selection)
commandes_prior_cibles = pd.concat(parties_prior, ignore_index=True)
ids_commandes_prior = set(commandes_prior_cibles["order_id"].tolist())

print(f"Commandes train disponibles : {len(commandes_train):,}")
print(f"Commandes cibles echantillonnees : {len(commandes_cibles):,}")
print(f"Commandes prior des clients retenus : {len(commandes_prior_cibles):,}")

Commandes train disponibles : 131,209
Commandes cibles echantillonnees : 20,000
Commandes prior des clients retenus : 313,328


## 2. Construire l'historique et les produits candidats

Les candidats comprennent les produits deja achetes par le client et les produits les plus frequents dans `prior`. On comparera l'ajout des 20, 50 ou 100 produits les plus populaires. Pour chaque candidat, on calcule le nombre d'achats precedents, la recence et des indicateurs de popularite globale. Le gros fichier est lu par morceaux.

In [9]:
agregats_clients_produits = []
agregats_produits = []
lignes_lues = 0
for numero_morceau, morceau in enumerate(pd.read_csv(
    DONNEES / "order_products__prior.csv",
    usecols=["order_id", "product_id", "reordered"],
    dtype={"order_id": "int32", "product_id": "int32", "reordered": "int8"},
    chunksize=TAILLE_MORCEAU,
)):
    lignes_lues += len(morceau)
    agregats_produits.append(
        morceau.groupby("product_id")["reordered"].agg(
            achats_prior="size", rachats_prior="sum"
        )
    )

    historique = morceau.loc[morceau["order_id"].isin(ids_commandes_prior)]
    if not historique.empty:
        historique = historique.merge(
            commandes_prior_cibles, on="order_id", how="left", validate="many_to_one"
        )
        agregats_clients_produits.append(
            historique.groupby(["user_id", "product_id"], as_index=False).agg(
                achats_historiques=("order_id", "size"),
                derniere_commande=("order_number", "max"),
                rachats_historiques=("reordered", "sum"),
            )
        )
    if (numero_morceau + 1) % 10 == 0:
        print(f"{lignes_lues:,} lignes prior lues...")

if not agregats_clients_produits:
    raise ValueError("Aucun produit historique trouve pour les clients selectionnes.")

historique_clients_produits = pd.concat(agregats_clients_produits, ignore_index=True)
historique_clients_produits = historique_clients_produits.groupby(
    ["user_id", "product_id"], as_index=False
).agg(
    achats_historiques=("achats_historiques", "sum"),
    derniere_commande=("derniere_commande", "max"),
    rachats_historiques=("rachats_historiques", "sum"),
)

statistiques_produits = pd.concat(agregats_produits).groupby(level=0).sum()
statistiques_produits["taux_rachat_produit"] = (
    statistiques_produits["rachats_prior"] / statistiques_produits["achats_prior"]
)
print(f"Paires client-produit candidates : {len(historique_clients_produits):,}")

5,000,000 lignes prior lues...
10,000,000 lignes prior lues...
15,000,000 lignes prior lues...
20,000,000 lignes prior lues...
25,000,000 lignes prior lues...
30,000,000 lignes prior lues...
Paires client-produit candidates : 1,295,485


## 3. Creer les etiquettes et mesurer la couverture

Pour elargir les choix au-dela des achats propres au client, on ajoute jusqu'aux 100 produits les plus frequents dans `prior`. Les pools top 20, top 50 et top 100 seront compares. Ces produits populaires proviennent uniquement des historiques, jamais des commandes cibles.

In [10]:
lignes_train = pd.read_csv(
    DONNEES / "order_products__train.csv",
    usecols=["order_id", "product_id"],
    dtype={"order_id": "int32", "product_id": "int32"},
)
lignes_cibles = lignes_train.loc[lignes_train["order_id"].isin(set(commandes_cibles["order_id"]))]

paires_historiques = historique_clients_produits.merge(
    commandes_cibles[["order_id", "user_id"]],
    on="user_id",
    how="inner",
    validate="many_to_one",
)
produits_historiques_attendus = paires_historiques.merge(
    lignes_cibles[["order_id", "product_id"]],
    on=["order_id", "product_id"],
    how="inner",
    validate="one_to_one",
)
couverture_historique = len(produits_historiques_attendus) / len(lignes_cibles)

NOMBRE_PRODUITS_POPULAIRES_MAX = 100
classement_populaire = statistiques_produits.nlargest(
    NOMBRE_PRODUITS_POPULAIRES_MAX, "achats_prior"
).reset_index()[["product_id", "achats_prior"]]
classement_populaire["rang_popularite"] = np.arange(1, len(classement_populaire) + 1)
ids_produits_populaires = classement_populaire["product_id"].to_numpy()
paires_populaires = pd.DataFrame({
    "order_id": np.repeat(commandes_cibles["order_id"].to_numpy(), len(ids_produits_populaires)),
    "user_id": np.repeat(commandes_cibles["user_id"].to_numpy(), len(ids_produits_populaires)),
    "product_id": np.tile(ids_produits_populaires, len(commandes_cibles)),
})
paires_candidates = pd.concat(
    [paires_historiques[["order_id", "user_id", "product_id"]], paires_populaires],
    ignore_index=True,
).drop_duplicates(["order_id", "product_id"])

candidats = paires_candidates.merge(
    historique_clients_produits,
    on=["user_id", "product_id"],
    how="left",
    validate="many_to_one",
).merge(
    commandes_cibles[
        ["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]
    ],
    on=["order_id", "user_id"],
    how="inner",
    validate="many_to_one",
)
candidats["produit_deja_achete"] = candidats["achats_historiques"].notna().astype("int8")
for colonne in ["achats_historiques", "derniere_commande", "rachats_historiques"]:
    candidats[colonne] = candidats[colonne].fillna(0)
candidats = candidats.merge(
    statistiques_produits[["achats_prior", "taux_rachat_produit"]],
    left_on="product_id",
    right_index=True,
    how="left",
    validate="many_to_one",
)
candidats = candidats.merge(
    classement_populaire[["product_id", "rang_popularite"]],
    on="product_id",
    how="left",
    validate="many_to_one",
)
candidats["rang_popularite"] = candidats["rang_popularite"].fillna(np.inf)
candidats = candidats.merge(
    lignes_cibles.assign(present_dans_commande=1),
    on=["order_id", "product_id"],
    how="left",
    validate="one_to_one",
)
candidats["present_dans_commande"] = (
    candidats["present_dans_commande"].fillna(0).astype("int8")
)
candidats["commandes_historiques_client"] = candidats["order_number"] - 1
candidats["frequence_achat_historique"] = (
    candidats["achats_historiques"] / candidats["commandes_historiques_client"].clip(lower=1)
)
candidats["commandes_depuis_dernier_achat"] = np.where(
    candidats["produit_deja_achete"].eq(1),
    candidats["commandes_historiques_client"] - candidats["derniere_commande"],
    candidats["commandes_historiques_client"],
)
candidats["log_achats_historiques"] = np.log1p(candidats["achats_historiques"])
candidats["log_popularite_produit"] = np.log1p(candidats["achats_prior"])
candidats["heure_sin"] = np.sin(2 * np.pi * candidats["order_hour_of_day"] / 24)
candidats["heure_cos"] = np.cos(2 * np.pi * candidats["order_hour_of_day"] / 24)
candidats["jour_sin"] = np.sin(2 * np.pi * candidats["order_dow"] / 7)
candidats["jour_cos"] = np.cos(2 * np.pi * candidats["order_dow"] / 7)

couverture = int(candidats["present_dans_commande"].sum()) / len(lignes_cibles)
resume_echantillon = pd.DataFrame({
    "indicateur": [
        "Commandes cibles", "Produits attendus", "Paires candidates",
        "Couverture avec historique uniquement", "Couverture avec les 100 produits populaires",
    ],
    "valeur": [
        len(commandes_cibles), len(lignes_cibles), len(candidats),
        couverture_historique, couverture,
    ],
})
display(resume_echantillon)

,indicateur,valeur
0,Commandes cibles,2.000000e+04
1,Produits attendus,2.123040e+05
2,Paires candidates,3.088091e+06
3,Couverture avec historique uniquement,5.953774e-01
4,Couverture avec les 100 produits populaires,6.546933e-01


## 4. Separer entrainement, validation et test

La separation se fait par commande cible; chaque client n'a qu'une commande `train` dans ce jeu. Le jeu de validation sert a choisir le seuil qui maximise le F1 en tenant compte aussi des produits attendus absents des candidats. Le jeu de test reste de cote jusqu'a l'evaluation finale.

In [12]:
ids = commandes_cibles["order_id"].to_numpy()
ids_entrainement, ids_temp = train_test_split(
    ids, test_size=0.30, random_state=GRAINE
)
ids_validation, ids_test = train_test_split(
    ids_temp, test_size=0.50, random_state=GRAINE
)

CARACTERISTIQUES = [
    "produit_deja_achete",
    "log_achats_historiques",
    "frequence_achat_historique",
    "commandes_depuis_dernier_achat",
    "taux_rachat_produit",
    "log_popularite_produit",
    "order_number",
    "days_since_prior_order",
    "heure_sin", "heure_cos", "jour_sin", "jour_cos",
]

def lignes_pour_commandes(ids_selectionnes):
    return candidats.loc[candidats["order_id"].isin(ids_selectionnes)]

entrainement = lignes_pour_commandes(ids_entrainement)
validation = lignes_pour_commandes(ids_validation)
test = lignes_pour_commandes(ids_test)
print(f"Paires candidates - entrainement : {len(entrainement):,}")
print(f"Paires candidates - validation : {len(validation):,}")
print(f"Paires candidates - test : {len(test):,}")

Paires candidates - entrainement : 2,163,398
Paires candidates - validation : 464,629
Paires candidates - test : 460,064


## 5. Entrainer une regression logistique

La cible vaut 1 lorsque le produit candidat est present dans la commande `train`, et 0 sinon. Un modele est entraine sur le pool jusqu'au top 100; on evalue ensuite les pools top 20, top 50 et top 100. Le seuil est choisi separement sur la validation en maximisant le F1 sur tous les produits attendus, puis les resultats sont calcules sur le test.

In [15]:
def calculer_mesures(cible, predictions, nombre_produits_attendus):
    precision = precision_score(cible, predictions, zero_division=0)
    vrais_positifs = int(np.sum((np.asarray(cible) == 1) & predictions))
    rappel_total = vrais_positifs / nombre_produits_attendus
    f1_total = (
        2 * precision * rappel_total / (precision + rappel_total)
        if precision + rappel_total > 0 else 0.0
    )
    return {
        "precision": precision,
        "rappel_parmi_candidats": recall_score(cible, predictions, zero_division=0),
        "rappel_tous_produits": rappel_total,
        "F1_tous_produits": f1_total,
        "part_candidates_recommandes": float(np.mean(predictions)),
    }

modele = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1_000, random_state=GRAINE),
)
modele.fit(entrainement[CARACTERISTIQUES], entrainement["present_dans_commande"])

lignes_attendues_validation = lignes_cibles.loc[
    lignes_cibles["order_id"].isin(ids_validation)
]
lignes_attendues_test = lignes_cibles.loc[lignes_cibles["order_id"].isin(ids_test)]
seuils = np.linspace(0.05, 0.95, 19)
resultats = []

historique_test = test.loc[test["produit_deja_achete"].eq(1)]
predictions_regle = historique_test["achats_historiques"].to_numpy() >= 2
mesures_regle = calculer_mesures(
    historique_test["present_dans_commande"],
    predictions_regle,
    len(lignes_attendues_test),
)
mesures_regle.update({
    "methode": "Regle historique seule (achat >= 2)",
    "produits_populaires_ajoutes": 0,
    "candidats_populaires_recommandes": 0,
    "couverture_test": historique_test["present_dans_commande"].sum() / len(lignes_attendues_test),
    "seuil_validation": np.nan,
})
resultats.append(mesures_regle)

for nombre_populaires in [20, 50, 100]:
    masque_validation = validation["produit_deja_achete"].eq(1) | validation["rang_popularite"].le(nombre_populaires)
    masque_test = test["produit_deja_achete"].eq(1) | test["rang_popularite"].le(nombre_populaires)
    validation_pool = validation.loc[masque_validation]
    test_pool = test.loc[masque_test]

    probabilites_validation = modele.predict_proba(validation_pool[CARACTERISTIQUES])[:, 1]
    scores_validation = [
        calculer_mesures(
            validation_pool["present_dans_commande"],
            probabilites_validation >= seuil,
            len(lignes_attendues_validation),
        )["F1_tous_produits"]
        for seuil in seuils
    ]
    seuil_retenu = float(seuils[int(np.argmax(scores_validation))])

    probabilites_test = modele.predict_proba(test_pool[CARACTERISTIQUES])[:, 1]
    mesures_modele = calculer_mesures(
        test_pool["present_dans_commande"],
        probabilites_test >= seuil_retenu,
        len(lignes_attendues_test),
    )
    mesures_modele.update({
        "methode": f"Modele - top {nombre_populaires}",
        "produits_populaires_ajoutes": nombre_populaires,
        "candidats_populaires_recommandes": int(
            ((test_pool["produit_deja_achete"].eq(0)) & (probabilites_test >= seuil_retenu)).sum()
        ),
        "couverture_test": test_pool["present_dans_commande"].sum() / len(lignes_attendues_test),
        "seuil_validation": seuil_retenu,
    })
    resultats.append(mesures_modele)

mesures = pd.DataFrame(resultats)
print("Le modele est entraine sur les candidats jusqu'au top 100; chaque seuil est choisi sur la validation.")
print("La couverture et le rappel total comptent les produits attendus absents des candidats comme non retrouves.")
display(mesures.round(3))

def predictions_hybrides(frame, probabilites, seuil, indices_nouveaux):
    predictions = pd.Series(probabilites >= seuil, index=frame.index)
    predictions.loc[indices_nouveaux] = True
    return predictions.to_numpy()

probabilites_validation_top100 = modele.predict_proba(validation[CARACTERISTIQUES])[:, 1]
probabilites_test_top100 = modele.predict_proba(test[CARACTERISTIQUES])[:, 1]
resultats_hybrides = []
for quota in [0, 1, 2, 3, 5]:
    indices_nouveaux_validation = (
        validation.loc[
            validation["produit_deja_achete"].eq(0), ["order_id", "rang_popularite"]
        ]
        .sort_values(["order_id", "rang_popularite"])
        .groupby("order_id")
        .head(quota)
        .index
    )
    indices_nouveaux_test = (
        test.loc[test["produit_deja_achete"].eq(0), ["order_id", "rang_popularite"]]
        .sort_values(["order_id", "rang_popularite"])
        .groupby("order_id")
        .head(quota)
        .index
    )
    meilleur_f1 = -1.0
    seuil_hybride = None
    for seuil in seuils:
        predictions_validation_hybrides = predictions_hybrides(
            validation, probabilites_validation_top100, seuil, indices_nouveaux_validation
        )
        score = calculer_mesures(
            validation["present_dans_commande"],
            predictions_validation_hybrides,
            len(lignes_attendues_validation),
        )["F1_tous_produits"]
        if score > meilleur_f1:
            meilleur_f1 = score
            seuil_hybride = float(seuil)

    predictions_test_hybrides = predictions_hybrides(
        test, probabilites_test_top100, seuil_hybride, indices_nouveaux_test
    )
    mesures_hybrides = calculer_mesures(
        test["present_dans_commande"],
        predictions_test_hybrides,
        len(lignes_attendues_test),
    )
    mesures_hybrides.update({
        "quota_nouveaux_par_commande": quota,
        "seuil_validation": seuil_hybride,
        "nouveaux_produits_recommandes_test": int(
            ((test["produit_deja_achete"].eq(0)) & predictions_test_hybrides).sum()
        ),
        "couverture_test": test["present_dans_commande"].sum() / len(lignes_attendues_test),
    })
    resultats_hybrides.append(mesures_hybrides)

mesures_hybrides = pd.DataFrame(resultats_hybrides)
print("Le quota est choisi sur la validation; le test reste reserve a l'evaluation finale.")
display(mesures_hybrides.round(3))

masque_historique_validation = validation["produit_deja_achete"].eq(1).to_numpy()
cible_validation = validation["present_dans_commande"].to_numpy()
meilleur_f1_separe = -1.0
seuil_historique_retenu = None
seuil_nouveau_retenu = None
for seuil_historique in seuils:
    for seuil_nouveau in seuils:
        predictions_validation = np.where(
            masque_historique_validation,
            probabilites_validation_top100 >= seuil_historique,
            probabilites_validation_top100 >= seuil_nouveau,
        )
        vrais_positifs = np.count_nonzero((cible_validation == 1) & predictions_validation)
        nombre_recommandations = np.count_nonzero(predictions_validation)
        precision = vrais_positifs / nombre_recommandations if nombre_recommandations else 0.0
        rappel = vrais_positifs / len(lignes_attendues_validation)
        f1 = 2 * precision * rappel / (precision + rappel) if precision + rappel else 0.0
        if f1 > meilleur_f1_separe:
            meilleur_f1_separe = f1
            seuil_historique_retenu = float(seuil_historique)
            seuil_nouveau_retenu = float(seuil_nouveau)

masque_historique_test = test["produit_deja_achete"].eq(1).to_numpy()
predictions_test_seuils_separes = np.where(
    masque_historique_test,
    probabilites_test_top100 >= seuil_historique_retenu,
    probabilites_test_top100 >= seuil_nouveau_retenu,
)
mesures_seuils_separes = calculer_mesures(
    test["present_dans_commande"],
    predictions_test_seuils_separes,
    len(lignes_attendues_test),
)
mesures_seuils_separes.update({
    "seuil_historique": seuil_historique_retenu,
    "seuil_nouveau_produit": seuil_nouveau_retenu,
    "nouveaux_produits_recommandes_test": int(
        ((test["produit_deja_achete"].eq(0)) & predictions_test_seuils_separes).sum()
    ),
})
print("Seuils separes appris sur la validation pour les produits historiques et nouveaux.")
display(pd.DataFrame([mesures_seuils_separes]).round(3))

Le modele est entraine sur les candidats jusqu'au top 100; chaque seuil est choisi sur la validation.
La couverture et le rappel total comptent les produits attendus absents des candidats comme non retrouves.


,precision,rappel_parmi_candidats,rappel_tous_produits,F1_tous_produits,part_candidates_recommandes,methode,produits_populaires_ajoutes,candidats_populaires_recommandes,couverture_test,seuil_validation
0,0.173,0.700,0.412,0.244,0.398,Regle historique seule (achat >= 2),0,0,0.589,NaN
1,0.283,0.584,0.356,0.315,0.167,Modele - top 20,20,0,0.609,0.75
2,0.283,0.566,0.356,0.315,0.125,Modele - top 50,50,0,0.628,0.75
3,0.283,0.548,0.356,0.315,0.087,Modele - top 100,100,0,0.649,0.75


Le quota est choisi sur la validation; le test reste reserve a l'evaluation finale.


,precision,rappel_parmi_candidats,rappel_tous_produits,F1_tous_produits,part_candidates_recommandes,quota_nouveaux_par_commande,seuil_validation,nouveaux_produits_recommandes_test,couverture_test
0,0.283,0.548,0.356,0.315,0.087,0,0.75,0,0.649
1,0.265,0.552,0.358,0.305,0.094,1,0.75,3000,0.649
2,0.250,0.555,0.361,0.295,0.100,2,0.75,6000,0.649
3,0.235,0.557,0.362,0.285,0.107,3,0.75,9000,0.649
4,0.211,0.562,0.365,0.268,0.120,5,0.75,15000,0.649


Seuils separes appris sur la validation pour les produits historiques et nouveaux.


,precision,rappel_parmi_candidats,rappel_tous_produits,F1_tous_produits,part_candidates_recommandes,seuil_historique,seuil_nouveau_produit,nouveaux_produits_recommandes_test
0,0.283,0.548,0.356,0.315,0.087,0.75,0.3,0


### Nouvelle tentative : un modele par type de candidat

Le modele unique ne retenait aucun produit populaire jamais achete. On entraine donc deux regressions logistiques distinctes : une pour les produits deja achetes, et une autre pour les produits populaires nouveaux. Les deux seuils sont choisis conjointement sur la validation, puis evalues une seule fois sur le test.

In [16]:
CARACTERISTIQUES_HISTORIQUE = [
    colonne for colonne in CARACTERISTIQUES if colonne != "produit_deja_achete"
]
CARACTERISTIQUES_NOUVEAU = [
    "log_popularite_produit", "taux_rachat_produit", "rang_popularite",
    "order_number", "days_since_prior_order",
    "heure_sin", "heure_cos", "jour_sin", "jour_cos",
]

entrainement_historique = entrainement.loc[entrainement["produit_deja_achete"].eq(1)]
entrainement_nouveaux = entrainement.loc[entrainement["produit_deja_achete"].eq(0)]
modele_historique = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1_000, random_state=GRAINE),
)
modele_nouveaux = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1_000, random_state=GRAINE),
)
modele_historique.fit(
    entrainement_historique[CARACTERISTIQUES_HISTORIQUE],
    entrainement_historique["present_dans_commande"],
)
if entrainement_nouveaux["present_dans_commande"].nunique() < 2:
    raise ValueError("L'echantillon ne contient pas les deux classes pour les nouveaux produits.")
modele_nouveaux.fit(
    entrainement_nouveaux[CARACTERISTIQUES_NOUVEAU],
    entrainement_nouveaux["present_dans_commande"],
)

def probabilites_specialisees(frame):
    deja_achetes = frame["produit_deja_achete"].eq(1).to_numpy()
    probabilites = np.empty(len(frame), dtype=float)
    achats = frame.loc[frame["produit_deja_achete"].eq(1)]
    nouveaux = frame.loc[frame["produit_deja_achete"].eq(0)]
    probabilites[deja_achetes] = modele_historique.predict_proba(
        achats[CARACTERISTIQUES_HISTORIQUE]
    )[:, 1]
    probabilites[~deja_achetes] = modele_nouveaux.predict_proba(
        nouveaux[CARACTERISTIQUES_NOUVEAU]
    )[:, 1]
    return probabilites

probabilites_validation_specialisees = probabilites_specialisees(validation)
probabilites_test_specialisees = probabilites_specialisees(test)
masque_historique_validation = validation["produit_deja_achete"].eq(1).to_numpy()
meilleur_f1_specialise = -1.0
seuil_historique_specialise = None
seuil_nouveau_specialise = None
for seuil_historique in seuils:
    for seuil_nouveau in seuils:
        predictions_validation = np.where(
            masque_historique_validation,
            probabilites_validation_specialisees >= seuil_historique,
            probabilites_validation_specialisees >= seuil_nouveau,
        )
        score = calculer_mesures(
            validation["present_dans_commande"],
            predictions_validation,
            len(lignes_attendues_validation),
        )["F1_tous_produits"]
        if score > meilleur_f1_specialise:
            meilleur_f1_specialise = score
            seuil_historique_specialise = float(seuil_historique)
            seuil_nouveau_specialise = float(seuil_nouveau)

masque_historique_test = test["produit_deja_achete"].eq(1).to_numpy()
predictions_test_specialisees = np.where(
    masque_historique_test,
    probabilites_test_specialisees >= seuil_historique_specialise,
    probabilites_test_specialisees >= seuil_nouveau_specialise,
)
mesures_specialisees = calculer_mesures(
    test["present_dans_commande"],
    predictions_test_specialisees,
    len(lignes_attendues_test),
)
mesures_specialisees.update({
    "seuil_historique": seuil_historique_specialise,
    "seuil_nouveau_produit": seuil_nouveau_specialise,
    "nouveaux_produits_recommandes_test": int(
        ((test["produit_deja_achete"].eq(0)) & predictions_test_specialisees).sum()
    ),
})
print("Evaluation des deux modeles specialises sur le test :")
display(pd.DataFrame([mesures_specialisees]).round(3))

Evaluation des deux modeles specialises sur le test :


,precision,rappel_parmi_candidats,rappel_tous_produits,F1_tous_produits,part_candidates_recommandes,seuil_historique,seuil_nouveau_produit,nouveaux_produits_recommandes_test
0,0.299,0.537,0.348,0.322,0.081,0.6,0.9,0


## 6. Interpreter le comparatif

Sur le test, la couverture passe de 58,9 % avec l'historique seul a 60,9 %, 62,8 % et 64,9 % en ajoutant respectivement les top 20, 50 et 100 produits populaires. Le F1 tous produits reste a 0,315 pour les trois pools, contre 0,244 pour la regle simple.

Le modele unique et les deux modeles specialises n'ont recommande aucun produit populaire jamais achete par le client, meme apres le reglage des seuils sur validation. La meilleure variante specialisee atteint un F1 de 0,322, mais reste fondee sur les achats historiques. Ajouter un quota de 1 a 5 produits populaires nouveaux augmente le rappel, mais fait baisser le F1 (de 0,305 a 0,268); cette option n'est donc pas retenue.

Conclusion pratique : l'elargissement augmente la couverture des produits candidats, mais les informations disponibles ne suffisent pas encore a recommander utilement des produits jamais achetes. Il vaut mieux conserver les recommandations historiques plutot que forcer ces produits populaires.

## 7. Examiner une commande test

L'exemple ci-dessous affiche les produits candidats les mieux notes pour une commande du jeu de test. La colonne `present_dans_commande` sert uniquement a comparer les recommandations avec la commande observee; elle n'a pas ete fournie au modele.

In [17]:
produits = pd.read_csv(DONNEES / "products.csv").set_index("product_id")["product_name"]
commande_exemple = int(ids_test[0])
exemple = test.loc[test["order_id"].eq(commande_exemple)].copy()
exemple["probabilite"] = modele.predict_proba(exemple[CARACTERISTIQUES])[:, 1]
exemple["nom_produit"] = exemple["product_id"].map(produits)
recommandations = exemple.sort_values("probabilite", ascending=False).head(10)
display(
    recommandations[["nom_produit", "probabilite", "present_dans_commande", "achats_historiques"]]
    .reset_index(drop=True)
)

,nom_produit,probabilite,present_dans_commande,achats_historiques
0,Large Grade AA Eggs,0.822448,1,2.0
1,Organic Whipped Naturally Buttery Spread,0.806850,0,2.0
2,Crackers,0.793037,1,2.0
3,Organic Avocado,0.776935,1,1.0
4,Large Lemon,0.758366,1,1.0
5,Limes,0.753104,1,1.0
6,Light Brown Sugar,0.746901,1,2.0
7,Buttermilk Complete Pancake & Waffle Mix,0.721388,0,2.0
8,2% Reduced Fat Milk,0.713478,0,1.0
9,Coke Classic,0.710054,1,1.0


## Conclusion et limites

Cette experience mesure la capacite a retrouver les produits de commandes `train` a partir des achats historiques et des produits populaires ajoutes. La precision est calculee parmi les produits recommandes. Le tableau distingue le rappel parmi les candidats du rappel sur tous les produits attendus; le F1 tous produits tient compte des produits encore absents des candidats comme non retrouves. La couverture permet de comparer les pools top 20, top 50 et top 100.

L'analyse utilise un echantillon reproductible de commandes `train`, et non les commandes `test` sans etiquettes. La regression logistique constitue un point de depart a comparer a la regle simple. Il faudra interpretrer les resultats avec prudence avant de presenter ce modele comme une solution de recommandation en production.